# Pothole Detection - Training with Motion Blur Augmented Dataset

This notebook downloads the augmented pothole dataset from Roboflow and trains a YOLOv8 model locally.

**Steps:**
1. Install dependencies
2. Download the augmented dataset from Roboflow
3. Train the YOLOv8 model
4. Validate the model
5. Export the best model

In [1]:
# Cell 1: Install Dependencies
!pip install roboflow ultralytics

Defaulting to user installation because normal site-packages is not writeable


In [2]:
# Cell 2: Download the Augmented Dataset from Roboflow
from roboflow import Roboflow

rf = Roboflow(api_key="6qUFlquza4a4asbRf8ml")
project = rf.workspace("ayushs-workspace-b1pqj").project("pothole-detection-bqu6s-z1sn6")
version = project.version(1)
dataset = version.download("yolov8")

print("\nDataset downloaded successfully!")
print("Dataset location:", dataset.location)

loading Roboflow workspace...
loading Roboflow project...

Dataset downloaded successfully!
Dataset location: c:\Users\DELL\OneDrive\Desktop\Pothole\Pothole-Detection--1


In [4]:
# Cell 3: Train the YOLOv8 Model
from ultralytics import YOLO
import os

# Option A: Start fresh from a pre-trained YOLOv8 Nano model (recommended for Raspberry Pi)
model = YOLO("yolov8n.pt")

# Option B: Continue training from your previous best model
# Uncomment the line below and update the path if you want to fine-tune your old model:
# model = YOLO(r"path\to\your\previous\best.pt")

# Path to the dataset config file
data_yaml_path = r"C:\Users\DELL\OneDrive\Desktop\Pothole\New folder\data.yaml"


# Start training!00

New https://pypi.org/project/ultralytics/8.4.22 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.21  Python-3.14.2 torch-2.10.0+cpu CPU (13th Gen Intel Core i5-13450HX)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\DELL\OneDrive\Desktop\Pothole\New folder\data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, mul

In [5]:
0# Cell 4: Validate the Model
#/.`,` This runs the trained model on the test set and shows accuracy metrics
metrics = model.val()

print("\n===== Validation Results =====")
print(f"mAP50:      {metrics.box.map50:.4f}")
print(f"mAP50-95:   {metrics.box.map:.4f}")
print(f"Precision:  {metrics.box.mp:.4f}")
print(f"Recall:     {metrics.box.mr:.4f}")

Ultralytics 8.4.21  Python-3.14.2 torch-2.10.0+cpu CPU (13th Gen Intel Core i5-13450HX)
Model summary (fused): 73 layers, 3,007,208 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 760.1121.9 MB/s, size: 88.6 KB)
val: Scanning C:\Users\DELL\OneDrive\Desktop\Pothole\New folder\valid\labels.cache... 491 images, 1 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 491/491 121.1Mit/s 0.0s
WARNING Box and segment counts should be equal, but got len(segments) = 6, len(boxes) = 1084. To resolve this only boxes will be used and all segments will be removed. To avoid this please supply either a detect or segment dataset, not a detect-segment mixed dataset.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 31/31 1.3s/it 40.8s1.1ss
                   all        491       1084      0.758      0.629      0.657      0.448
   circle-drain-clean-          4          4      0.796      0.982      0.796       0.56
       

In [6]:
# Cell 5: Test on a sample image (optional)
import glob

# Find a test image from the dataset
test_images = glob.glob(os.path.join(dataset.location, "test", "images", "*"))

if test_images:
    sample_image = test_images[0]
    print(f"Running inference on: {sample_image}")
    results = model.predict(source=sample_image, save=True, conf=0.25)
    print("Result saved in runs/detect/predict/")
else:
    print("No test images found. You can manually test with:")
    print('model.predict(source="path/to/your/image.jpg", save=True, conf=0.25)')

No test images found. You can manually test with:
model.predict(source="path/to/your/image.jpg", save=True, conf=0.25)


## Next Steps

After training is complete:

1. Your best model is saved at: `runs/detect/pothole_blur_augmented/weights/best.pt`
2. Copy this `best.pt` file to your Raspberry Pi's `raspberry_pi/models/` folder
3. Update `config.py` on the Raspberry Pi to point to the new model
4. Test the detection with blurred/motion images to verify improvement!